In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
MODEL_NAME = "../model/Qwen2.5-7B-Instruct"  # 模型名称

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME, 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
import wandb
wandb.login(key="1b8adc705fb9b3e125c05f15107ad7c22c830811")
wandb.init(project="Qwen2.5-7B-Instruct-Continued-Pretraining")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",

                      "embed_tokens", "lm_head",], # Add for continual pretraining
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
from datasets import Dataset
# 1. 读取整个 txt 文件为字符串
with open("../data-continued-pretraining/news_6month.txt", "r", encoding="utf-8") as f:
    full_text = f.read()

# 2. 用 <|endoftext|> 分割为样本
samples = [s.strip() for s in full_text.split("<|endoftext|>") if s.strip()]

# 3. 构造 Dataset 对象
dataset = Dataset.from_dict({"text": samples})

EOS_TOKEN = "<|im_end|>"
def formatting_prompts_func(examples):
    return { "text" : [example + EOS_TOKEN for example in examples["text"]] }
dataset = dataset.map(formatting_prompts_func, batched = True,)

In [ ]:
print(dataset)

In [ ]:
for row in dataset[:4]["text"]:
    print("=========================")
    print(row)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth import UnslothTrainer, UnslothTrainingArguments

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 8,

    args = UnslothTrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,

        warmup_ratio = 0.1,
        num_train_epochs = 3,

        learning_rate = 5e-5,
        embedding_learning_rate = 5e-6,

        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.00,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "../output-continued-pretraining/",
        report_to = "wandb", # Use this for WandB etc
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
from transformers import TextIteratorStreamer
from threading import Thread
text_streamer = TextIteratorStreamer(tokenizer)
import textwrap
max_print_width = 100

# Before running inference, call `FastLanguageModel.for_inference` first

FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    "中国的首都是什么"
]*1, return_tensors = "pt").to("cuda")

generation_kwargs = dict(
    inputs,
    streamer = text_streamer,
    max_new_tokens = 1024,
    use_cache = True,
)
thread = Thread(target = model.generate, kwargs = generation_kwargs)
thread.start()

length = 0
for j, new_text in enumerate(text_streamer):
    if j == 0:
        wrapped_text = textwrap.wrap(new_text, width = max_print_width)
        if len(wrapped_text) > 0:
            length = len(wrapped_text[-1])
        else:
            length = 0
        wrapped_text = "\n".join(wrapped_text)
        print(wrapped_text, end = "")
    else:
        length += len(new_text)
        if length >= max_print_width:
            length = 0
            print()
        print(new_text, end = "")
    pass
pass

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
base_model = AutoModelForCausalLM.from_pretrained(
    "../model/Qwen2.5-7B-Instruct",
    torch_dtype="auto",    # 通常会是 float16
    device_map="auto"
)
# 加载 LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    "../output-continued-pretraining/3years/checkpoint-1980",
)

In [ ]:
for name, param in list(model.named_parameters())[:10]:
    print(f"{name}: dtype={param.dtype}, shape={param.shape}")

In [ ]:
# 合并 LoRA 权重
model = model.merge_and_unload()

# 保存合并后的完整模型
model.save_pretrained("../model/Qwen2.5-7B-Instruct-CPT-3years")
tokenizer = AutoTokenizer.from_pretrained("../model/Qwen2.5-7B-Instruct")
tokenizer.save_pretrained("../model/Qwen2.5-7B-Instruct-CPT-3years")